# 优惠券是否增加消费？完整 DRE / AIPW 学习项目
**Renee 的逐步学习版 · 模拟观察性研究**

问题：如果给目标人群中的所有用户发券，与都不发券相比，人均消费改变多少？
目标为总体 ATE = E[Y(1) − Y(0)]。时间顺序：历史消费和活跃度 → 发券 → 后续消费。

依次完成：数据生成 → 数据检查 → 五折交叉拟合 → PS 评估 → overlap / 权重 / balance → outcome 评估 → ATE 与区间 → 模型设错实验 → 低 overlap 压力实验 → 结论。

运行：Jupyter 中选择 **Restart Kernel and Run All**。依赖：`pip install numpy pandas scikit-learn matplotlib jupyter`。
这里沿用已学的 LogisticRegression 与 LinearRegression，不引入复杂调参。真实项目若按这些诊断反复挑选模型，应使用嵌套验证或保留独立评估数据。

## 1. 先声明因果假设
1. 无未观测混杂：给定历史消费和活跃度后，发券与潜在消费结果条件独立。在模拟中由生成机制保证；真实业务需要领域知识，不能用模型分数证明。
2. Positivity / overlap：每种目标背景下，发券与不发券都有机会发生。
3. 一致性与无干扰：券的定义明确，一个用户的发券不改变其他用户的消费。
4. 用户是独立抽样单位，协变量来自发券之前。真实数据若一个用户多行，或存在时间依赖，要按用户 / 时间切分并重新设计推断。

PS 是 propensity score；PSM 专指 matching。本项目采用 IPW 和 AIPW，没有执行匹配。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, mean_absolute_error, mean_squared_error
from sklearn.calibration import calibration_curve

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)
rng = np.random.default_rng(42)
n = 10000
past_spend = rng.uniform(0, 200, n)
activity = rng.integers(0, 11, n)
log_odds = -2 + 0.01 * past_spend + 0.2 * activity
true_ps = 1 / (1 + np.exp(-log_odds))
treatment = rng.binomial(1, true_ps)
baseline = 20 + 0.5 * past_spend + 5 * activity
true_effect = 10 + 2 * activity
noise = rng.normal(0, 10, n)
outcome = baseline + treatment * true_effect + noise
df = pd.DataFrame({'past_spend': past_spend, 'activity': activity,
                   'treatment': treatment, 'outcome': outcome})
true_sample_ate = true_effect.mean()
true_population_ate = 20.0
print(df.head().round(2))
print('Sample true effect:', round(true_sample_ate, 3), '| Population ATE:', true_population_ate)

## 2. 数据质量和原始差异
模拟中每行就是一位用户，没有重复用户 ID 问题。实际数据需额外检查唯一 ID、发券时间、消费观察窗口、重复发券、缺失机制及处理定义。
原始组间差同时包含券的作用和用户背景差异。

In [ ]:
assert not df.isna().any().any()
assert set(df['treatment'].unique()) == {0, 1}
assert np.isfinite(df.to_numpy()).all()
print(df.groupby('treatment').agg(n=('outcome', 'size'), past_spend=('past_spend', 'mean'),
      activity=('activity', 'mean'), outcome=('outcome', 'mean')).round(3))
naive_difference = df.loc[df.treatment == 1, 'outcome'].mean() - df.loc[df.treatment == 0, 'outcome'].mean()
print('Naive difference:', round(naive_difference, 3))

## 3. 五折 cross-fitting：每个人的预测不使用自己的标签
每轮：用 80% 用户训练 PS；再分别用训练折 T=0 和 T=1 的用户训练两个 outcome 模型；对剩下 20% 的每个人产生 ps、mu0、mu1。
还保存“只预测训练折均值”的简单基线，避免只看一个 MAE 就说模型好。
以下封装可重复使用；核心仍是熟悉的 `.fit()` 和 `.predict()`。

In [ ]:
def cross_fit(data, ps_cols, outcome_cols, seed=42):
    T = data['treatment'].to_numpy()
    Y = data['outcome'].to_numpy()
    pred = {name: np.full(len(data), np.nan) for name in
            ['ps', 'mu0', 'mu1', 'base_ps', 'base0', 'base1']}
    counts = np.zeros(len(data), dtype=int)
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for train_idx, valid_idx in folds.split(data, T):
        train = data.iloc[train_idx]
        valid = data.iloc[valid_idx]
        ps_model = Pipeline([('scaler', StandardScaler()),
                             ('model', LogisticRegression(max_iter=1000))])
        ps_model.fit(train[ps_cols], train['treatment'])
        pred['ps'][valid_idx] = ps_model.predict_proba(valid[ps_cols])[:, 1]
        pred['base_ps'][valid_idx] = train['treatment'].mean()
        for state in [0, 1]:
            group = train.loc[train['treatment'] == state]
            model = LinearRegression()
            model.fit(group[outcome_cols], group['outcome'])
            pred[f'mu{state}'][valid_idx] = model.predict(valid[outcome_cols])
            pred[f'base{state}'][valid_idx] = group['outcome'].mean()
        counts[valid_idx] += 1
    assert (counts == 1).all()  # 每个人只被作为验证用户预测一次
    assert all(np.isfinite(v).all() for v in pred.values())
    return pred

features = ['past_spend', 'activity']
pred = cross_fit(df, features, features)
for name in ['ps', 'mu0', 'mu1']:
    df[name] = pred[name]
print(df[['treatment', 'outcome', 'ps', 'mu0', 'mu1']].head().round(3))

## 4. PS 的预测质量：概率是否合理？
ROC-AUC 衡量区分能力，**不是越高因果估计就越好**，也不能单独衡量 overlap。
Brier 和 log loss 越小越好，但同时受校准、区分能力和数据本身随机性影响，不能单独当作纯校准指标。
校准表：预测约 0.8 的用户，实际是否约 80% 收到券？每个分箱都要看样本量。
这里所有预测和基线都是样本外的。

In [ ]:
T = df['treatment'].to_numpy()
Y = df['outcome'].to_numpy()
ps = df['ps'].to_numpy()
ps_metrics = pd.DataFrame([
    {'model': name, 'ROC_AUC': roc_auc_score(T, p), 'Brier': brier_score_loss(T, p),
     'Log_loss': log_loss(T, p)}
    for name, p in [('OOF PS', ps), ('OOF constant baseline', pred['base_ps'])]])
print(ps_metrics.round(4))
calibration = pd.DataFrame({'ps': ps, 'T': T})
calibration['bin'] = pd.qcut(ps, 10, duplicates='drop')
print(calibration.groupby('bin', observed=True).agg(n=('T', 'size'), predicted=('ps', 'mean'),
      observed=('T', 'mean')).round(3))
observed, predicted = calibration_curve(T, ps, n_bins=10, strategy='quantile')
plt.figure(figsize=(6, 4))
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.plot(predicted, observed, 'o-', label='OOF PS')
plt.xlabel('Mean predicted probability'); plt.ylabel('Observed treatment rate')
plt.title('PS calibration'); plt.legend(); plt.tight_layout(); plt.show()

## 5. Overlap：相似背景是否有两种状态？
PS 直方图是总体诊断，不证明每个协变量区域都有对照，因此同时查看“历史消费 × 活跃度”的网格人数。
0.05 / 0.95 是提醒检查尾部的示例阈值，不是统一合格线。不自动删除任何用户。

In [ ]:
bins = np.linspace(0, 1, 11)
ps_bins = pd.cut(ps, bins=bins, include_lowest=True)
overlap_counts = pd.crosstab(ps_bins, T).reindex(columns=[0, 1], fill_value=0)
print('Counts by PS bin:\n', overlap_counts)
print('Share with PS outside [0.05, 0.95]:', round(np.mean((ps < .05) | (ps > .95)), 4))
spend_bin = pd.cut(df.past_spend, bins=[0, 50, 100, 150, 200], include_lowest=True)
activity_bin = pd.cut(df.activity, bins=[-1, 3, 7, 10])
support_grid = pd.crosstab([spend_bin, activity_bin], df.treatment).reindex(columns=[0, 1], fill_value=0)
print('Covariate grid counts (coarse diagnostic only):\n', support_grid)
plt.figure(figsize=(7, 4))
for state, label in [(0, 'No coupon'), (1, 'Coupon')]:
    plt.hist(ps[T == state], bins=np.linspace(0, 1, 21), density=True, alpha=.5, label=label)
plt.xlabel('OOF propensity score'); plt.ylabel('Density'); plt.legend()
plt.title('Treatment / control overlap'); plt.tight_layout(); plt.show()

## 6. 权重、ESS 和 SMD：加权是否稳定、是否平衡？
权重：T=1 时 1/ps；T=0 时 1/(1−ps)。这里不做 clipping。
有效样本量 ESS = (Σw)² / Σw²：权重越集中，ESS 越小；这是权重集中程度的诊断，不是精确功效计算。
SMD 分母固定为加权前两组方差平均值的平方根，使前后可比较。|SMD|<0.1 是常用经验参考，不是保证因果正确的证明。
增加平方项与交互项诊断，以免只看均值就忽略部分分布差异。

In [ ]:
def weights_from_ps(t, p):
    if not np.all((p > 0) & (p < 1)):
        raise ValueError('PS contains 0 or 1: inspect support/model before estimating.')
    return np.where(t == 1, 1 / p, 1 / (1 - p))

def weight_diagnostics(t, w):
    rows = []
    for state in [0, 1]:
        a = w[t == state]
        ess = a.sum() ** 2 / np.square(a).sum()
        rows.append({'T': state, 'n': len(a), 'mean': a.mean(), 'p99': np.quantile(a, .99),
                     'max': a.max(), 'ESS': ess, 'ESS_fraction': ess / len(a),
                     'top_1pct_weight_share': np.sort(a)[-max(1, int(np.ceil(.01*len(a)))):].sum()/a.sum()})
    return pd.DataFrame(rows)

def balance_diagnostics(data, t, w):
    check = data[features].copy()
    check['past_spend_squared'] = data.past_spend ** 2
    check['activity_squared'] = data.activity ** 2
    check['spend_x_activity'] = data.past_spend * data.activity
    rows = []
    for col in check:
        a = check.loc[t == 1, col].to_numpy()
        b = check.loc[t == 0, col].to_numpy()
        sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
        wa = np.average(a, weights=w[t == 1])
        wb = np.average(b, weights=w[t == 0])
        rows.append({'feature': col, 'treated_before': a.mean(), 'control_before': b.mean(),
                     'treated_after': wa, 'control_after': wb,
                     'SMD_before': (a.mean()-b.mean())/sd if sd else np.nan,
                     'SMD_after': (wa-wb)/sd if sd else np.nan})
    return pd.DataFrame(rows)

w = weights_from_ps(T, ps)
df['weight'] = w
weight_table = weight_diagnostics(T, w)
balance_table = balance_diagnostics(df, T, w)
print(weight_table.round(3))
print(balance_table.round(3))
plt.figure(figsize=(7, 4))
plt.plot(balance_table.SMD_before.abs(), balance_table.feature, 'o', label='Before')
plt.plot(balance_table.SMD_after.abs(), balance_table.feature, 'o', label='After')
plt.axvline(.1, ls='--', color='gray', label='0.1 reference')
plt.xlabel('Absolute SMD'); plt.title('Covariate balance'); plt.legend(); plt.tight_layout(); plt.show()

## 7. Outcome 模型：只在真实发生的状态验证
对实际 T=0 用户，用 mu0 对照实际 Y；对实际 T=1 用户，用 mu1 对照实际 Y。
不能拿 T=1 用户的实际消费验证 mu0：那是不同状态。
MAE 是平均绝对误差；RMSE 更重罚大误差；残差 = 实际 − 预测。没有通用“低于多少就能做因果”的界线。
同时按活跃度和历史消费分箱看平均残差；总体均值接近零，仍可能掩盖不同人群的系统偏差。

In [ ]:
outcome_rows = []
for state in [0, 1]:
    mask = T == state
    for label, values in [('Outcome model', pred[f'mu{state}']),
                          ('Training-group mean baseline', pred[f'base{state}'])]:
        outcome_rows.append({'state': state, 'model': label, 'n': mask.sum(),
            'MAE': mean_absolute_error(Y[mask], values[mask]),
            'RMSE': np.sqrt(mean_squared_error(Y[mask], values[mask])),
            'mean_residual': (Y[mask]-values[mask]).mean()})
outcome_metrics = pd.DataFrame(outcome_rows)
print(outcome_metrics.round(3))
factual_prediction = np.where(T == 1, pred['mu1'], pred['mu0'])
residuals = df[['treatment', 'past_spend', 'activity']].copy()
residuals['residual'] = Y - factual_prediction
for col in features:
    residuals['bin'] = pd.qcut(residuals[col], 4, duplicates='drop')
    print('Residuals by', col)
    print(residuals.groupby(['treatment', 'bin'], observed=True).agg(
        n=('residual', 'size'), mean_residual=('residual', 'mean'),
        residual_sd=('residual', 'std')).round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for state, ax in zip([0, 1], axes):
    mask = T == state
    ax.scatter(factual_prediction[mask], (Y-factual_prediction)[mask], s=5, alpha=.15)
    ax.axhline(0, color='black', ls='--')
    ax.set(xlabel='OOF prediction', ylabel='Actual - prediction', title=f'Factual residuals: T={state}')
plt.tight_layout(); plt.show()

## 8. 估计 ATE：四种方法与置信区间
归一化 IPW（Hájek）：两组分别计算 ΣwY / Σw，再相减。
AIPW：平均 [mu1−mu0 + T(Y−mu1)/ps − (1−T)(Y−mu0)/(1−ps)]。
dr_score 是用户对总体估计的贡献，不是已经识别出的个人真实效果。

这里的近似 95% 区间针对总体 ATE，使用 iid 用户、cross-fitting、基线正确设定且满足常规矩条件的影响函数标准误。
不能把“任一模型正确即可一致”理解成“任一模型正确时，这个简单区间都自动有效”；真实项目需要检查 nuisance 收敛条件与抽样结构。
模拟样本真实均值约 20.07；总体真值为 20，两者不要混淆。

In [ ]:
def aipw_scores(y, t, p, m0, m1):
    weights_from_ps(t, p)  # 检查概率合法，拒绝悄悄截断
    return m1-m0 + t*(y-m1)/p - (1-t)*(y-m0)/(1-p)

ipw_ate = np.average(Y[T == 1], weights=w[T == 1]) - np.average(Y[T == 0], weights=w[T == 0])
df['predicted_effect'] = df.mu1 - df.mu0
df['correction_1'] = T * (Y-df.mu1) / ps
df['correction_0'] = (1-T) * (Y-df.mu0) / (1-ps)
df['dr_score'] = aipw_scores(Y, T, ps, pred['mu0'], pred['mu1'])
dr_ate = df.dr_score.mean()
se = df.dr_score.std(ddof=1) / np.sqrt(n)
ci_low, ci_high = dr_ate - 1.96*se, dr_ate + 1.96*se
ate_table = pd.DataFrame({'method': ['Naive', 'Normalized IPW', 'Outcome only', 'AIPW'],
    'estimate': [naive_difference, ipw_ate, df.predicted_effect.mean(), dr_ate]})
ate_table['error_vs_sample_truth'] = ate_table.estimate - true_sample_ate
print(ate_table.round(3))
print(f'AIPW population-ATE approximate 95% CI: [{ci_low:.3f}, {ci_high:.3f}]; SE={se:.3f}')
print('Mean correction:', round((df.correction_1-df.correction_0).mean(), 5))

## 9. 双重稳健性实验：故意漏掉 activity
固定同一份数据、同样的切分，只改变模型输入。
这是真值已知的教学实验；真实业务里不能用未知的真实 ATE 来挑模型。
PS 正确或两个 outcome 条件均值模型正确，可在其他必要条件下得到一致估计；两者都错没有保证。
一次模拟结果是示例，不是普遍证明；这里不对设错模型套用上一节的简单区间。

In [ ]:
bad_outcome = cross_fit(df, features, ['past_spend'])
bad_ps = cross_fit(df, ['past_spend'], features)
both_bad = cross_fit(df, ['past_spend'], ['past_spend'])
scenarios = [('Both full', pred), ('PS full, outcome omits activity', bad_outcome),
             ('PS omits activity, outcome full', bad_ps), ('Both omit activity', both_bad)]
scenario_rows = []
for label, p in scenarios:
    value = aipw_scores(Y, T, p['ps'], p['mu0'], p['mu1']).mean()
    scenario_rows.append({'scenario': label, 'outcome_only': (p['mu1']-p['mu0']).mean(),
                          'AIPW': value, 'AIPW_error_vs_sample_truth': value-true_sample_ate})
scenario_table = pd.DataFrame(scenario_rows)
print(scenario_table.round(3))
print('Bad-PS weighted balance:')
print(balance_diagnostics(df, T, weights_from_ps(T, bad_ps['ps']))[
    ['feature', 'SMD_before', 'SMD_after']].round(3))
print('Bad-outcome factual MAE by state:')
for state in [0, 1]:
    mask = T == state
    print(state, round(mean_absolute_error(Y[mask], bad_outcome[f'mu{state}'][mask]), 3))

## 10. 压力实验：不是换模型就能修好的 overlap 问题
新的发券机制让低历史消费几乎全不发券、高历史消费几乎全发券。
只改变 treatment / outcome 的生成，保留用户背景；这属于另一个模拟研究。
展示 PS 尾部比例、ESS、最大权重、稀少对照区域和 outcome 外推风险。
结构性零概率时，数据无法非参数识别全体 ATE；即使模型给出数值，也不等于有证据支持。
下面是接近零的概率，不是严格零，有限样本仍可能没有对照。

In [ ]:
stress = df[['past_spend', 'activity']].copy()
stress_rng = np.random.default_rng(2026)
stress_true_ps = 1/(1+np.exp(-0.15*(stress.past_spend.to_numpy()-100)))
stress_t = stress_rng.binomial(1, stress_true_ps)
stress['treatment'] = stress_t
stress['outcome'] = baseline + stress_t*true_effect + stress_rng.normal(0, 10, n)
stress_pred = cross_fit(stress, features, features)
stress_w = weights_from_ps(stress_t, stress_pred['ps'])
print('PS outside [0.05, 0.95]:', round(np.mean((stress_pred['ps']<.05)|(stress_pred['ps']>.95)), 3))
print(weight_diagnostics(stress_t, stress_w).round(3))
print(pd.crosstab(pd.cut(stress.past_spend, [0, 50, 100, 150, 200], include_lowest=True), stress_t))
plt.figure(figsize=(7, 4))
for state in [0, 1]:
    plt.hist(stress_pred['ps'][stress_t == state], bins=np.linspace(0, 1, 21), alpha=.5,
             density=True, label=f'T={state}')
plt.title('Stress scenario: poor overlap'); plt.xlabel('OOF PS'); plt.ylabel('Density')
plt.legend(); plt.tight_layout(); plt.show()
print('Do not treat a computed full-population estimate as reliable without support.')

## 11. 诊断记录：提醒，不是自动发放“模型合格证”
下面只汇总事实和经验阈值触发项。尾部 PS、ESS 和 SMD 的阈值需结合业务、目标人群、样本量和敏感性分析预先设定。
模型表现不能证明无未观测混杂，均值平衡不能证明联合分布完全平衡。

In [ ]:
diagnostics = pd.Series({
    'PS Brier': ps_metrics.loc[0, 'Brier'],
    'PS baseline Brier': ps_metrics.loc[1, 'Brier'],
    'Share PS outside [0.05,0.95]': np.mean((ps<.05)|(ps>.95)),
    'Max absolute weighted SMD (checked terms)': balance_table.SMD_after.abs().max(),
    'Minimum ESS fraction': weight_table.ESS_fraction.min(),
    'Maximum weight': w.max(),
    'AIPW estimate': dr_ate,
    'AIPW approximate CI low': ci_low,
    'AIPW approximate CI high': ci_high,
})
print(diagnostics.round(4))
print('Review terms with |SMD| >= 0.1:',
      balance_table.loc[balance_table.SMD_after.abs() >= .1, 'feature'].tolist())

## 12. 如何写项目结论
运行结果应支持以下层次的陈述：
- 原始差异受到选择偏差影响；在这个模拟中能用已知真值验证这一点。
- 完整 PS 的样本外概率质量与基线比较；结合校准图解读，不用 AUC 判定因果模型好坏。
- 两组存在较广 overlap；所检查的特征、平方项与交互项加权后更平衡。尾部、ESS 和稀少区域需同时查看。
- 两个 outcome 模型分别在事实状态下验证，并和训练折均值基线比较；反事实准确性无法直接从真实标签验证。
- 报告 AIPW 估计与总体 ATE 区间。消费提升不等于利润提升，实际决策还要减券成本并考虑毛利。
- 双重稳健实验说明设错其中一侧时的表现；不承诺有限样本中 DRE 永远更准。
- 低 overlap 时优先补数据或做随机实验；限制人群会改变 estimand；截断权重只是偏差与方差的取舍，不能补出缺失对照。

### 真实数据迁移清单
明确单位 / treatment / 时间零点 / 结果窗口 / 目标人群；制定混杂变量集，排除处理后变量；按用户或时间划分；检查缺失、重复及人群变化；完成本项目诊断；选择与抽样结构相符的推断；加入业务可行的负对照或对未观测混杂的敏感性分析。它们提供补充证据，不是识别假设的证明。

### 自测
1. 为什么高 AUC 不能证明 IPW 适用？
2. 为什么 model0 不能用收到券用户的实际消费验证？
3. 两个模型都漏掉同一个混杂变量时，DRE 还保证正确吗？
4. 若删掉无 overlap 人群，最终 ATE 对应谁？

### 资料
- [scikit-learn: Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)
- [DoWhy: Support and overlap](https://www.pywhy.org/dowhy/main/example_notebooks/dowhy_refuter_assess_overlap.html)
- [Funk et al., 2011: Doubly Robust Estimation of Causal Effects](https://pmc.ncbi.nlm.nih.gov/articles/PMC3070495/)

本项目是完整的模拟教学研究；未提供实际优惠券数据，因此不包含真实业务因果结论或已完成的真实数据敏感性验证。